# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel
import numpy as np
import pandas as pd

import os

WRITE_DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
CHECKPOINT_DIR = os.environ.get("FRAUD_GRAPH_CHECKPOINT_DIR", "file:///tmp/onboarding_fraud/anomaly_ckpt")
spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
SCORE_TBL = f"{WRITE_DB}.{EMP}_v2_final_scores"
EDGES_TXN = f"{WRITE_DB}.{EMP}_v2_edges_transferred"
OWN_TBL   = f"{WRITE_DB}.{EMP}_v2_edges_ownership"
SUBG_NIDX = f"{WRITE_DB}.{EMP}_v2_subgraph_node_index"

ANCHOR_TRAILING = {
    "202401": ["202310", "202311", "202312"],
    "202402": ["202311", "202312", "202401"],
    "202403": ["202312", "202401", "202402"],
}
TEST_ONE_ANCHOR = False
RARE_BENEF_MAX = int(os.environ.get("FRAUD_GRAPH_RARE_BENEF_MAX", "25"))
K_PCTS = [float(value) for value in os.environ.get("FRAUD_GRAPH_EVALUATION_PCTS", "0.02,0.05").split(",")]
ISO_EPS  = 1e-9

RUN_UNANCHORED_FRAUDAR = True
MIN_RARE_OUT = int(os.environ.get("FRAUD_GRAPH_MIN_RARE_OUT", "3"))
FRAUDAR_EPS = float(os.environ.get("FRAUD_GRAPH_FRAUDAR_EPSILON", "0.5"))
FRAUDAR_MAX_ROUNDS = int(os.environ.get("FRAUD_GRAPH_FRAUDAR_MAX_ROUNDS", "15"))
FRAUDAR_MAX_BLOCKS = int(os.environ.get("FRAUD_GRAPH_FRAUDAR_MAX_BLOCKS", "5"))
FRAUDAR_MIN_ACCT = int(os.environ.get("FRAUD_GRAPH_FRAUDAR_MIN_ACCOUNTS", "2"))
FRAUDAR_MAX_ACCT = int(os.environ.get("FRAUD_GRAPH_FRAUDAR_MAX_ACCOUNTS", "2500"))

OUT_EGO   = f"{WRITE_DB}.{EMP}_v2_anomaly_features"
OUT_VALID = f"{WRITE_DB}.{EMP}_v2_anomaly_validation"

spark.conf.set("spark.sql.shuffle.partitions", os.environ.get("FRAUD_GRAPH_SHUFFLE_PARTITIONS", "200"))

def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

anchors = (["202401"] if TEST_ONE_ANCHOR else list(ANCHOR_TRAILING.keys()))
print("anomaly | anchors:", anchors, "| rare<=", RARE_BENEF_MAX, "| un-anchored Fraudar:", RUN_UNANCHORED_FRAUDAR)


In [ ]:
def ego_features(anchor):
    months = ANCHOR_TRAILING[anchor]
    e0 = (spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months))
          .select("src", "dst", F.col("sum_amount").cast("double").alias("amt"))
          .where(F.col("src") != F.col("dst")).persist(StorageLevel.DISK_ONLY))
    e0.count()
    fanin = e0.groupBy("dst").agg(F.countDistinct("src").alias("fi"))
    rare_dst = fanin.where(F.col("fi") <= RARE_BENEF_MAX).select("dst")
    out_f = e0.groupBy("src").agg(F.countDistinct("dst").alias("out_deg"),
                                  F.sum("amt").alias("out_amt"), F.count("*").alias("out_txn"))
    in_f  = e0.groupBy("dst").agg(F.countDistinct("src").alias("in_deg"), F.sum("amt").alias("in_amt"))
    rare_out = (e0.join(rare_dst, "dst").groupBy("src")
                .agg(F.countDistinct("dst").alias("n_rare_benef"), F.sum("amt").alias("rare_out_amt")))
    acct = (out_f.select(F.col("src").alias("id"), "out_deg", "out_amt", "out_txn")
            .join(in_f.select(F.col("dst").alias("id"), "in_deg", "in_amt"), "id", "outer")
            .join(rare_out.select(F.col("src").alias("id"), "n_rare_benef", "rare_out_amt"), "id", "outer")
            .na.fill(0.0))
    own = spark.table(OWN_TBL).select(F.col("dst").alias("id"), F.col("src").alias("cust"))
    cust_f = (acct.join(own, "id", "inner").groupBy("cust").agg(
                  F.max("out_deg").alias("out_deg"), F.max("in_deg").alias("in_deg"),
                  F.sum("out_amt").alias("out_amt"), F.sum("in_amt").alias("in_amt"),
                  F.max("out_txn").alias("out_txn"),
                  F.max("n_rare_benef").alias("n_rare_benef"), F.sum("rare_out_amt").alias("rare_out_amt"))
              .withColumn("anchor_month", F.lit(anchor)))
    e0.unpersist()
    return cust_f

first = True
for a in anchors:
    print("=== ego features:", a, "===")
    cf = ego_features(a)
    if first:
        save_overwrite(cf, OUT_EGO); first = False
    else:
        cf.write.mode("append").saveAsTable(OUT_EGO)
    print("   saved ego features for", a, "->", OUT_EGO)
print("ego feature table:", OUT_EGO, "| rows:", spark.table(OUT_EGO).count())


In [ ]:
def _densest_block(bip, eps, max_rounds):
    S_acct = bip.select("src").distinct(); S_mule = bip.select("dst").distinct()
    best_g = -1.0; best_acct = None; best_mule = None
    for r in range(max_rounds):
        eis = bip.join(S_acct, "src", "left_semi").join(S_mule, "dst", "left_semi")
        f = eis.agg(F.sum("cw").alias("f")).first()["f"]
        if not f or f <= 0: break
        ns = S_acct.count() + S_mule.count()
        if ns == 0: break
        g = float(f) / ns
        if g > best_g:
            best_g = g
            if best_acct is not None: best_acct.unpersist()
            if best_mule is not None: best_mule.unpersist()
            best_acct = S_acct.persist(StorageLevel.DISK_ONLY); best_acct.count()
            best_mule = S_mule.persist(StorageLevel.DISK_ONLY); best_mule.count()
        thr = 2.0 * (1.0 + eps) * g
        S_acct = eis.groupBy("src").agg(F.sum("cw").alias("d")).where(F.col("d") > thr).select("src").checkpoint(eager=True)
        S_mule = eis.groupBy("dst").agg(F.sum("cw").alias("d")).where(F.col("d") > thr).select("dst").checkpoint(eager=True)
        if S_acct.limit(1).count() == 0 or S_mule.limit(1).count() == 0: break
    return best_acct, best_mule, best_g

def unanchored_fraudar(anchor):
    months = ANCHOR_TRAILING[anchor]
    e0 = (spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months))
          .select("src", "dst").where(F.col("src") != F.col("dst")).distinct())
    fanin = e0.groupBy("dst").agg(F.countDistinct("src").alias("fi"))
    rare = fanin.where(F.col("fi") <= RARE_BENEF_MAX).select("dst", "fi")
    er = e0.join(rare.select("dst"), "dst", "inner")
    ring_acct = er.groupBy("src").agg(F.countDistinct("dst").alias("rout")).where(F.col("rout") >= MIN_RARE_OUT).select("src")
    w = rare.withColumn("cw", 1.0 / F.log2(F.col("fi") + 5.0)).select("dst", "cw")
    bip = (er.join(ring_acct, "src", "left_semi").join(w, "dst")
           .select("src", "dst", "cw").persist(StorageLevel.DISK_ONLY))
    bip.count()
    blocks = []
    for blk in range(FRAUDAR_MAX_BLOCKS):
        ba, bm, bg = _densest_block(bip, FRAUDAR_EPS, FRAUDAR_MAX_ROUNDS)
        if ba is None: break
        nacct = ba.count()
        if nacct < FRAUDAR_MIN_ACCT:
            ba.unpersist(); bm.unpersist(); break
        keep = (nacct <= FRAUDAR_MAX_ACCT)
        if keep:
            blocks.append(ba.select(F.col("src").alias("id")).withColumn("ublock_density", F.lit(float(bg))))
        print("   unanchored block %d: %d accounts, density=%.4f%s"
              % (blk, nacct, bg, "" if keep else "  (too big -> peeled past, NOT flagged)"))
        bip2 = bip.join(ba, "src", "left_anti").join(bm, "dst", "left_anti").persist(StorageLevel.DISK_ONLY)
        bip2.count(); bip.unpersist(); bip = bip2
        ba.unpersist(); bm.unpersist()
    bip.unpersist()
    if not blocks:
        return None
    allb = blocks[0]
    for bdf in blocks[1:]:
        allb = allb.unionByName(bdf)
    own = spark.table(OWN_TBL).select(F.col("dst").alias("id"), F.col("src").alias("cust"))
    return (allb.join(own, "id", "inner").groupBy("cust")
            .agg(F.max("ublock_density").alias("ublock_density"))
            .withColumn("in_dense_block_unanchored", F.lit(1)).withColumn("anchor_month", F.lit(anchor)))

fraud_cust = None
if RUN_UNANCHORED_FRAUDAR:
    for a in anchors:
        print("=== un-anchored Fraudar:", a, "===")
        try:
            fc = unanchored_fraudar(a)
        except Exception as fe:
            print("   [WARN] un-anchored Fraudar failed for", a, "->", fe); fc = None
        if fc is not None:
            fraud_cust = fc if fraud_cust is None else fraud_cust.unionByName(fc)
    if fraud_cust is not None:
        fraud_cust = fraud_cust.persist(StorageLevel.DISK_ONLY); fraud_cust.count()
        print("un-anchored Fraudar flagged customers:", fraud_cust.count())
    else:
        print("un-anchored Fraudar flagged no small dense blocks.")


In [ ]:
base = spark.table(SCORE_TBL).select("cust", "anchor_month", "target_fraud", "final_score")
ego  = spark.table(OUT_EGO)
full = base.join(ego, ["cust", "anchor_month"], "left")
if fraud_cust is not None:
    full = full.join(fraud_cust.select("cust", "anchor_month", "in_dense_block_unanchored", "ublock_density"),
                     ["cust", "anchor_month"], "left")
else:
    full = full.withColumn("in_dense_block_unanchored", F.lit(0)).withColumn("ublock_density", F.lit(0.0))

EGO_COLS = ["out_deg", "in_deg", "out_amt", "in_amt", "out_txn", "n_rare_benef", "rare_out_amt",
            "in_dense_block_unanchored", "ublock_density"]
full = full.na.fill({c: 0 for c in EGO_COLS}).na.fill({"final_score": 0.0})
pdf = full.select("cust", "anchor_month", "target_fraud", "final_score", *EGO_COLS).toPandas()
pdf["target_fraud"] = pdf["target_fraud"].fillna(0).astype(int)
for c in EGO_COLS + ["final_score"]:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0.0)
ANCHORS = sorted(pdf["anchor_month"].unique().tolist())

pdf["star_ratio"]   = pdf["out_deg"] / (pdf["out_deg"] + pdf["in_deg"] + 1.0)
pdf["rare_ratio"]   = pdf["n_rare_benef"] / (pdf["out_deg"] + 1.0)
pdf["passthrough"]  = np.minimum(pdf["in_amt"], pdf["out_amt"]) / (np.maximum(pdf["in_amt"], pdf["out_amt"]) + 1.0)
pdf["velocity"]     = pdf["out_txn"] / (pdf["out_deg"] + 1.0)
print("master frame:", pdf.shape, "| anchors:", ANCHORS,
      "| bad:", {a: int(pdf.loc[pdf.anchor_month == a, "target_fraud"].sum()) for a in ANCHORS})


In [ ]:
ANOMALY_FEATS = ["out_deg", "n_rare_benef", "rare_ratio", "star_ratio", "passthrough", "velocity",
                 "in_dense_block_unanchored"]

def capture_at_pd(y, score, ks):
    y = np.asarray(y, dtype=int); score = np.asarray(score, dtype=float)
    order = np.argsort(-score, kind="mergesort"); ys = y[order]; n = len(y); nbad = int(y.sum()); rows = []
    for p in ks:
        K = max(1, int(n * p)); tb = int(ys[:K].sum())
        cap = (tb / nbad) if nbad else 0.0
        lift = ((tb / K) / (nbad / n)) if nbad else 0.0
        rows.append((p, K, tb, cap, lift))
    return rows, n, nbad

def _pctrank_group(s):
    n = len(s)
    if n <= 1: return pd.Series(np.zeros(n), index=s.index)
    return (s.rank(method="min") - 1.0) / (n - 1.0)

def _woe_iv(cnt, bad):
    good = cnt - bad; tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6) if tb else bad * 0 + 1e-6
    pg = (good / tg).clip(lower=1e-6) if tg else good * 0 + 1e-6
    return float(((pb - pg) * np.log(pb / pg)).sum())

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

PR = {c: pdf.groupby("anchor_month")[c].transform(_pctrank_group).fillna(0.0).values for c in ANOMALY_FEATS}
pdf["anomaly_score"] = np.mean([PR[c] for c in ANOMALY_FEATS], axis=0)


In [ ]:
y_all = pdf["target_fraud"].values
print("=== per-anomaly-feature IV (pooled) + full-population capture ===")
for f in ANOMALY_FEATS + ["anomaly_score"]:
    b = _robust_bins(pdf[f].values, max_bins=10)
    g = pd.DataFrame({"b": b, "y": y_all}).groupby("b").agg(cnt=("y", "size"), bad=("y", "sum"))
    iv = _woe_iv(g["cnt"], g["bad"])
    rows, n, nbad = capture_at_pd(y_all, pdf[f].values, [0.05, 0.10])
    print("   %-26s IV=%.3f  cap@5%%=%.4f (lift %.2f)  cap@10%%=%.4f (lift %.2f)"
          % (f, iv, rows[0][3], rows[0][4], rows[1][3], rows[1][4]))

valid_rows = []
for scope, mask in [("FULL", np.ones(len(pdf), dtype=bool))]:
    rows, n, nbad = capture_at_pd(y_all[mask], pdf.loc[mask, "anomaly_score"].values, K_PCTS)
    for (p, K, tb, cap, lift) in rows:
        valid_rows.append(("anomaly_score", scope, p, K, n, nbad, tb, round(cap, 4), round(lift, 3)))


In [ ]:
iso = (pdf["final_score"].values <= ISO_EPS)
iso_y = y_all[iso]; iso_score = pdf.loc[iso, "anomaly_score"].values
n_iso = int(iso.sum()); nbad_iso = int(iso_y.sum())
print("=== ISOLATED set: %d customers, %d frauds (base rate %.3f%%) ===" % (n_iso, nbad_iso, 100.0 * nbad_iso / max(n_iso, 1)))
print("   (frauds pooled: %d isolated of %d total = %.1f%% graph-isolated)"
      % (nbad_iso, int(y_all.sum()), 100.0 * nbad_iso / max(int(y_all.sum()), 1)))
rows, n, nbad = capture_at_pd(iso_y, iso_score, K_PCTS)
best_iso_lift = 0.0
for (p, K, tb, cap, lift) in rows:
    valid_rows.append(("anomaly_score", "ISOLATED", p, K, n, nbad, tb, round(cap, 4), round(lift, 3)))
    best_iso_lift = max(best_iso_lift, lift)
    print("   isolated  top%4.0f%%  K=%d  captured %d/%d isolated frauds  capture=%.4f  lift=%.2f"
          % (p * 100, K, tb, nbad_iso, cap, lift))

if "in_dense_block_unanchored" in pdf.columns:
    fl = (pdf["in_dense_block_unanchored"].values > 0)
    fl_iso = fl & iso
    print("   un-anchored Fraudar flags %d isolated customers, of which %d are fraud (precision %.1f%%)"
          % (int(fl_iso.sum()), int(y_all[fl_iso].sum()), 100.0 * y_all[fl_iso].sum() / max(int(fl_iso.sum()), 1)))

save_overwrite(spark.createDataFrame(valid_rows, ["score", "scope", "top_pct", "K", "n", "n_bad",
                                                  "bad_in_topK", "capture", "lift"]), OUT_VALID)
print("saved validation:", OUT_VALID)


In [ ]:
print("=== Notebook 09 (v2) summary ===")
print("ego/anomaly features:", OUT_EGO, "| validation:", OUT_VALID)
print("Full-population anomaly capture@K and the ISOLATED-set capture@K are saved above.")
review_threshold = float(os.environ.get("FRAUD_GRAPH_ANOMALY_REVIEW_LIFT", "1.25"))
VERDICT = ("candidate for further validation" if best_iso_lift >= review_threshold else
           "limited incremental signal in this sample")
print("VERDICT: best isolated-set lift = %.2f -> unsupervised anomaly %s." % (best_iso_lift, VERDICT))
print("Interpret this exploratory diagnostic alongside approved non-graph baselines and holdout tests.")
print("No production decision should be made from this stage without independent validation.")
print("NOTEBOOK 09 COMPLETE — anomaly features and validation diagnostics generated.")
if fraud_cust is not None:
    fraud_cust.unpersist()
